# Chapter 1 — A one-zone model from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/chemical-evolution-tutorial/blob/main/notebooks/01_one_zone.ipynb)

**Goal.** Write the simplest model of a galaxy's chemical evolution in a few lines of numpy, and check it against
exact solutions. The galaxy is a single, well-mixed box of gas (a *one-zone* model). Gas flows in, some of it turns
into stars, the stars return some gas and make new metals, and a wind blows gas out. Everything in later chapters,
VICE included, solves these same equations with fewer approximations.

**You will learn**
1. the equations of a one-zone model: inflow, star formation, recycling, outflow;
2. how to integrate them numerically, and how to check the integration (conservation, exact solutions, step size);
3. why the metallicity of the gas approaches an **equilibrium**, set by the yield and the outflow;
4. which parameter sets *where* the metallicity ends up and which sets *how fast* it gets there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1.1 The equations

Track three masses: the gas $M_g$, the metals in the gas $M_Z$ (for one element, say an α-element made only by
core-collapse supernovae), and the stars formed. Time $t$ is in Gyr, masses in $M_\odot$, rates in $M_\odot\,$Gyr$^{-1}$.

- **Star formation.** The star-formation rate is proportional to the gas mass,
  $\dot M_\star = M_g/\tau_\star$, where $\tau_\star$ is the *depletion time* (its inverse is the star-formation
  efficiency, SFE).
- **Recycling.** Stars return a fraction $r$ of their mass to the gas as they die. Here we make the *instantaneous
  recycling* approximation: the return happens at once, at the gas's own metallicity. For a Kroupa initial mass
  function, $r \approx 0.4$ (the value VICE's documentation recommends).
- **Outflow.** Supernovae drive a wind that removes gas at a rate $\eta\,\dot M_\star$; $\eta$ is the *mass-loading
  factor*. The wind has the metallicity of the gas.
- **Yield.** Each solar mass of stars formed makes $y$ solar masses of new metals at once (core-collapse supernovae
  explode within ~40 Myr of star formation, short compared with the Gyr time scales of a galaxy).

The gas and its metals then obey

$$\dot M_g = \dot M_{\rm in} - (1 + \eta - r)\,\frac{M_g}{\tau_\star}, \qquad
  \dot M_Z = y\,\frac{M_g}{\tau_\star} - (1 + \eta - r)\,\frac{M_Z}{\tau_\star},$$

with metal-free inflow. Both have the same loss term: gas is removed by star formation and outflow and partly
returned by recycling, $(1+\eta-r)$ in total per unit of star formation. This defines the **depletion time of the
gas**, $\tau_{\rm dep} = \tau_\star / (1+\eta-r)$, the time scale on which a gas reservoir reacts.

We integrate with the simplest scheme, **Euler steps**: evaluate the rates at time $t$, and advance every mass by
rate × $\Delta t$.

In [ ]:
def one_zone(t_end, dt, inflow, tau_star, eta, r=0.4, y=0.015, Mg0=0.0):
    """Euler integration of the one-zone model with instantaneous recycling.

    inflow: function of t [Gyr] returning the inflow rate [Msun/Gyr]. Returns a dict of arrays on t = 0, dt, ..., t_end:
    gas mass Mg, metal mass Mz, metallicity Z = Mz/Mg, cumulative inflow Min, stars formed Mform, outflow Mout [Msun].
    """
    n = int(round(t_end / dt))
    t = np.arange(n + 1) * dt
    Mg, Mz, Min, Mform, Mout = (np.zeros(n + 1) for _ in range(5))
    Mg[0] = Mg0
    for i in range(n):
        sfr = Mg[i] / tau_star                               # star-formation rate, Msun/Gyr
        z = Mz[i] / Mg[i] if Mg[i] > 0 else 0.0              # metallicity of the gas (0 before any gas exists)
        f_in = inflow(t[i])
        Mg[i + 1] = Mg[i] + dt * (f_in - (1 + eta - r) * sfr)          # gas equation
        Mz[i + 1] = Mz[i] + dt * (y * sfr - (1 + eta - r) * sfr * z)  # metal equation
        Min[i + 1] = Min[i] + dt * f_in                                # bookkeeping, for the conservation check
        Mform[i + 1] = Mform[i] + dt * sfr
        Mout[i + 1] = Mout[i] + dt * eta * sfr
    Z = np.divide(Mz, Mg, out=np.zeros_like(Mz), where=Mg > 0)
    return dict(t=t, Mg=Mg, Mz=Mz, Z=Z, Min=Min, Mform=Mform, Mout=Mout)


# A galaxy that starts with no gas and receives 1e9 Msun/Gyr of metal-free gas for 10 Gyr
P = dict(tau_star=2.0, eta=2.5, r=0.4, y=0.015)
m = one_zone(10.0, 0.001, inflow=lambda t: 1e9, **P)
tau_dep = P['tau_star'] / (1 + P['eta'] - P['r'])
print(f"tau_dep = {tau_dep:.3f} Gyr;  at 10 Gyr: Mg = {m['Mg'][-1]:.3e}, stars formed = {m['Mform'][-1]:.3e}, "
      f"outflow = {m['Mout'][-1]:.3e} Msun")

**Check 1: mass is conserved.** Every solar mass that flowed in is now gas, in long-lived stars and remnants
(formed mass minus the returned fraction $r$), or in the wind. With Euler steps this holds to rounding error,
because every term is computed from the same star-formation rate.

In [ ]:
budget = m['Mg'] + (1 - P['r']) * m['Mform'] + m['Mout']
print(f"max |inflow - (gas + locked + wind)| / inflow = {np.max(np.abs(m['Min'] - budget)[1:] / m['Min'][1:]):.1e}")
assert np.allclose(budget, m['Min'], rtol=1e-12, atol=1e-3)    # identity: mass conservation

## 1.2 Two exact solutions

**The gas.** For a constant inflow $\dot M_{\rm in}$ and no initial gas, the gas equation is linear with constant
coefficients:

$$M_g(t) = \dot M_{\rm in}\,\tau_{\rm dep}\,\left(1 - e^{-t/\tau_{\rm dep}}\right).$$

The gas grows on the time scale $\tau_{\rm dep}$ to a steady value where inflow balances the losses.

**The metals.** Insert $M_g(t)$ into the metal equation. With $x = t/\tau_{\rm dep}$ and the solution
$M_Z = (y\dot M_{\rm in}\tau_{\rm dep}^2/\tau_\star)\,[1 - e^{-x} - x e^{-x}]$ (substitute it to check), the
metallicity $Z = M_Z/M_g$ is

$$Z(t) = Z_{\rm eq}\left[1 - \frac{x\,e^{-x}}{1 - e^{-x}}\right], \qquad Z_{\rm eq} = \frac{y}{1+\eta-r}.$$

**A second case: constant star formation.** If the galaxy starts with gas $M_{g,0}$ and the inflow exactly
replaces what is lost, $\dot M_{\rm in} = (1+\eta-r)M_{g,0}/\tau_\star$, the gas mass and the star-formation rate
stay constant, and

$$Z(t) = Z_{\rm eq}\left(1 - e^{-t/\tau_{\rm dep}}\right).$$

In both cases $Z$ approaches the same **equilibrium metallicity** $Z_{\rm eq} = y/(1+\eta-r)$: new metals made per
unit of star formation ($y$) balance the metals removed per unit of star formation ($1+\eta-r$ times $Z$). Weinberg,
Andrews & Freudenburg (2017, https://doi.org/10.3847/1538-4357/837/2/183) derive these and many more one-zone
solutions.

In [ ]:
def gas_exact(t, f_in, tau_dep):
    return f_in * tau_dep * (1 - np.exp(-t / tau_dep))


def Z_exact_inflow(t, Zeq, tau_dep):
    x = np.maximum(t / tau_dep, 1e-12)                  # avoid 0/0 at t = 0 (the limit is Z = 0)
    return Zeq * (1 - x * np.exp(-x) / -np.expm1(-x))   # expm1(-x) = e^-x - 1, accurate for small x


def Z_exact_constant(t, Zeq, tau_dep):
    return Zeq * (1 - np.exp(-t / tau_dep))


Zeq = P['y'] / (1 + P['eta'] - P['r'])
Mg0 = 1e9
mc = one_zone(10.0, 0.001, inflow=lambda t: (1 + P['eta'] - P['r']) * Mg0 / P['tau_star'], Mg0=Mg0, **P)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(m['t'], m['Mg'] / 1e9, color='#2a78d6', lw=2, label='gas (numerical)')
ax.plot(m['t'], (1 - P['r']) * m['Mform'] / 1e9, color='#eb6834', lw=2, label='stars and remnants')
ax.plot(m['t'], m['Mout'] / 1e9, color='#1baf7a', lw=2, label='wind')
ax.plot(m['t'][::500], gas_exact(m['t'][::500], 1e9, tau_dep) / 1e9, 'ko', ms=4, label='gas (exact)')
ax.set(xlabel='time [Gyr]', ylabel=r'mass [$10^9\,M_\odot$]')
ax.legend(frameon=False, fontsize=9)
ax = axes[1]
ax.plot(m['t'], m['Z'] / Zeq, color='#2a78d6', lw=2, label='constant inflow, no initial gas')
ax.plot(mc['t'], mc['Z'] / Zeq, color='#e87ba4', lw=2, label='constant star formation')
ax.plot(m['t'][::250], Z_exact_inflow(m['t'][::250], Zeq, tau_dep) / Zeq, 'ko', ms=4, label='exact')
ax.plot(mc['t'][::250], Z_exact_constant(mc['t'][::250], Zeq, tau_dep) / Zeq, 'ko', ms=4)
ax.axhline(1, color='0.5', lw=0.8, ls=':')
ax.set(xlabel='time [Gyr]', ylabel=r'$Z / Z_{\rm eq}$', xlim=(0, 5))
ax.legend(frameon=False, fontsize=9)
for ax in axes:
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

err_g = np.max(np.abs(m['Mg'][1:] / gas_exact(m['t'][1:], 1e9, tau_dep) - 1))
err_z = np.max(np.abs(m['Z'][100:] - Z_exact_inflow(m['t'][100:], Zeq, tau_dep))) / Zeq
err_c = np.max(np.abs(mc['Z'] - Z_exact_constant(mc['t'], Zeq, tau_dep))) / Zeq
print(f'max relative error: gas {err_g:.1e};  Z (inflow, t > 0.1 Gyr) {err_z:.1e};  Z (constant SFR) {err_c:.1e}')
assert err_g < 2e-3 and err_z < 2e-3 and err_c < 2e-3      # limit: exact solutions, at dt = 1 Myr

**Figure 1.1.** Left: mass budget of a galaxy that starts without gas and receives $10^9\,M_\odot\,$Gyr$^{-1}$ of
metal-free gas ($\tau_\star = 2$ Gyr, $\eta = 2.5$, $r = 0.4$): gas (blue), mass locked in stars and remnants
(orange) and mass in the wind (green), in $10^9\,M_\odot$, against time; black points: the exact gas mass. Right:
gas metallicity in units of $Z_{\rm eq}$ for this galaxy (blue) and for one with constant star formation (pink);
black points: the exact solutions.

The numerical model follows the exact solutions to better than 0.2%. The gas mass levels off after a few
$\tau_{\rm dep} = 0.65$ Gyr, and so does the metallicity. The constant-SFR galaxy enriches faster at first: it starts
with a full gas reservoir forming stars, whereas the other galaxy must first accumulate its gas.

**Check 3: the step size.** Euler's method is *first order*: its error should halve when $\Delta t$ halves. If it
did not, the code would have a bug that the exact solutions might not reveal at a single $\Delta t$.

In [ ]:
dts = np.array([0.04, 0.02, 0.01, 0.005, 0.0025])
errs = []
for dt in dts:
    mm = one_zone(5.0, dt, inflow=lambda t: 1e9, **P)
    errs.append(np.max(np.abs(mm['Mg'][1:] / gas_exact(mm['t'][1:], 1e9, tau_dep) - 1)))
errs = np.array(errs)
slope = np.polyfit(np.log(dts), np.log(errs), 1)[0]

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.loglog(dts * 1e3, errs, 'o-', color='#2a78d6', lw=2, label='Euler, gas mass')
ax.loglog(dts * 1e3, errs[0] * dts / dts[0], 'k--', lw=1, label=r'$\propto \Delta t$')
ax.set(xlabel=r'time step $\Delta t$ [Myr]', ylabel='max relative error of $M_g$')
ax.legend(frameon=False)
ax.grid(color='0.9', lw=0.6, which='both')
print(f'error vs step: slope {slope:.3f}')
assert abs(slope - 1) < 0.05                                 # convergence: first order

**Figure 1.2.** Largest relative error of the numerical gas mass (over 0–5 Gyr, against the exact solution) as a
function of the time step, for the constant-inflow galaxy of Figure 1.1. Dashed: a line proportional to
$\Delta t$.

The error falls in proportion to $\Delta t$ (slope 1.00): the integration behaves as Euler's method should. A step
of 1 Myr is far shorter than $\tau_{\rm dep}$ and gives errors of ~0.1%.

## 1.3 What sets the equilibrium, and how fast it is reached

The two parameters of star formation and feedback play different roles:
- $\eta$ enters $Z_{\rm eq} = y/(1+\eta-r)$: a stronger wind carries away more metals and lowers the equilibrium.
- $\tau_\star$ does not enter $Z_{\rm eq}$ at all. It sets only the time scale $\tau_{\rm dep}$: a galaxy with
  inefficient star formation reaches the same metallicity, later.

Figure 1.3 shows both, in logarithmic units relative to an adopted reference value $Z_{\rm ref} = 0.0057$ (close to the
Sun's oxygen mass fraction), so that the y-axis reads like [O/H].

In [ ]:
Zref = 0.0057
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for eta, col in zip((0.5, 2.5, 6.0), ('#2a78d6', '#1baf7a', '#eb6834')):
    mm = one_zone(10.0, 0.002, inflow=lambda t: 1e9, **{**P, 'eta': eta})
    axes[0].plot(mm['t'][2:], np.log10(mm['Z'][2:] / Zref), color=col, lw=2, label=fr'$\eta$ = {eta}')
    axes[0].axhline(np.log10(P['y'] / (1 + eta - P['r']) / Zref), color=col, lw=0.8, ls=':')
for ts, col in zip((0.5, 2.0, 8.0), ('#2a78d6', '#1baf7a', '#eb6834')):
    mm = one_zone(10.0, 0.002, inflow=lambda t: 1e9, **{**P, 'tau_star': ts})
    axes[1].plot(mm['t'][2:], np.log10(mm['Z'][2:] / Zref), color=col, lw=2, label=fr'$\tau_*$ = {ts} Gyr')
    t_half = mm['t'][np.argmax(mm['Z'] > 0.5 * Zeq)]
    print(f'tau_star = {ts}: Z at 10 Gyr = {mm["Z"][-1] / Zeq:.3f} Zeq; reaches Zeq/2 at {t_half:.2f} Gyr')
axes[1].axhline(np.log10(Zeq / Zref), color='0.4', lw=0.8, ls=':')
axes[0].set_ylabel(r'$\log_{10}(Z / Z_{\rm ref})$  (like [O/H])')
for ax in axes:
    ax.set(xlabel='time [Gyr]', ylim=(-1.5, 0.6))
    ax.legend(frameon=False, fontsize=9)
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

drop = np.log10((1 + 6.0 - P['r']) / (1 + 0.5 - P['r']))
assert abs(drop - 0.78) < 0.005                         # identity: the eta = 0.5 -> 6 drop quoted below
# the equilibrium depends on eta, not on tau_star
for ts in (0.5, 8.0):
    mm = one_zone(30.0, 0.005, inflow=lambda t: 1e9, **{**P, 'tau_star': ts})
    assert abs(mm['Z'][-1] / Zeq - 1) < 0.01            # limit: Z -> y/(1+eta-r) for any tau_star

**Figure 1.3.** Gas metallicity, $\log_{10}(Z/Z_{\rm ref})$ with $Z_{\rm ref} = 0.0057$, against time for the
constant-inflow galaxy. Left: three mass-loading factors $\eta$ at $\tau_\star = 2$ Gyr; dotted lines: their
$Z_{\rm eq}$. Right: three depletion times $\tau_\star$ at $\eta = 2.5$; dotted line: the common $Z_{\rm eq}$.

The outflow sets the level: going from $\eta = 0.5$ to 6 lowers the equilibrium by
$\log_{10}(6.6/1.1) = 0.78$ dex. The star-formation
efficiency sets the pace: all three galaxies on the right reach the same metallicity, at times differing by more
than a factor of 10 (printed above). Dwarf galaxies are metal-poor because their winds are strong (large $\eta$),
and their metallicities rise slowly when their star formation is inefficient (long $\tau_\star$).

This model has a single element. To see the [α/Fe] knee of Chapter 0 we need a second element, iron, made with a
delay by type Ia supernovae: Chapter 2.

## Summary

- A one-zone model is two linear equations, for the gas and its metals; star formation, recycling and outflow enter
  only through $\tau_\star$ and the combination $1+\eta-r$.
- The numerical model conserves mass exactly, matches two exact solutions to < 0.2% at $\Delta t$ = 1 Myr, and its
  error falls in proportion to $\Delta t$.
- The metallicity approaches $Z_{\rm eq} = y/(1+\eta-r)$ on the time scale $\tau_{\rm dep} = \tau_\star/(1+\eta-r)$:
  the wind sets the level, star-formation efficiency the pace.

## Exercises

1. Give the inflow an exponential decline, $\dot M_{\rm in} \propto e^{-t/\tau_{\rm in}}$ with $\tau_{\rm in}$ = 1
   Gyr. Does $Z$ still approach $Z_{\rm eq}$? (Weinberg et al. 2017, §3, discuss the answer.)
2. Make the inflow metal-rich, with metallicity $Z_{\rm in}$. Add the term to the metal equation and find the new
   equilibrium.
3. Replace Euler steps by the midpoint method (evaluate the rates at $t+\Delta t/2$). Repeat Figure 1.2: what slope do
   you expect?